# Thai Food Classification Training (EfficientNetB0)

**Objective**: Train a 97%+ accuracy Thai food classifier using EfficientNetB0 transfer learning, then export to TensorFlow.js for browser deployment.

**Dataset**: 10 Thai food classes, ~305 images (30-31 per class), perfectly balanced

**Key Improvements**:
- ✅ Uses EfficientNetB0 (more efficient than MobileNetV2)
- ✅ TensorFlow 2.x compatible (avoids Keras 3.x export issues)
- ✅ Includes TensorFlow.js export step
- ✅ Clean data augmentation (zoom + horizontal flip)
- ✅ Proper callbacks (checkpoint, early stopping, learning rate reduction)

In [ ]:
# Import Libraries
import tensorflow as tf
import numpy as np
from numpy.random import seed
seed(1)
from tensorflow.random import set_seed
set_seed(2)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import pandas as pd
import matplotlib.pyplot as plt
import os
import cv2
import json
import zipfile
from pathlib import Path

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
## Step 0: Upload the Dataset Zip

Use the file picker to upload `thai_food_dataset.zip` into the current working directory.
# Upload zip file via widget (local Jupyter/VS Code) or Colab
import os
from pathlib import Path

zip_name = "thai_food_dataset.zip"

# Detect Colab
try:
    import google.colab
    from google.colab import files
    print("Colab detected. Please choose the zip file to upload...")
    uploaded = files.upload()
    if zip_name not in uploaded:
        print(f"WARNING: Expected {zip_name}, got: {list(uploaded.keys())}")
except Exception:
    try:
        import ipywidgets as widgets
        from IPython.display import display

        print("Local/VS Code detected. Use the file picker below to upload the zip.")
        uploader = widgets.FileUpload(accept='.zip', multiple=False)
        display(uploader)

        def _save_upload(change):
            if not uploader.value:
                return
            file_info = next(iter(uploader.value.values()))
            with open(zip_name, 'wb') as f:
                f.write(file_info['content'])
            print(f"Saved: {zip_name} ({os.path.getsize(zip_name)/1024/1024:.2f} MB)")

        uploader.observe(_save_upload, names='value')
    except Exception as e:
        print("Could not open file picker.")
        print("Place thai_food_dataset.zip in the notebook folder and continue.")
        print(f"Reason: {e}")

print(f"Exists: {Path(zip_name).exists()}")


## Step 1: Extract and Prepare Dataset

In [ ]:
# Extract dataset from zip if needed
dataset_path = Path('dataset')
zip_path = Path('thai_food_dataset.zip')

if not dataset_path.exists() and zip_path.exists():
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall()
    print("Extraction complete!")

# Verify dataset structure
if dataset_path.exists():
    classes = sorted([d for d in dataset_path.iterdir() if d.is_dir()])
    print(f"\nFound {len(classes)} classes:")
    for cls in classes:
        img_count = len(list(cls.glob('*.jpg'))) + len(list(cls.glob('*.jpeg'))) + len(list(cls.glob('*.png')))
        print(f"  - {cls.name}: {img_count} images")
else:
    print("ERROR: Dataset not found. Make sure thai_food_dataset.zip is in the current directory")

In [ ]:
# Create train/validation/test split (70/15/15) from single dataset
from sklearn.model_selection import train_test_split

base_path = 'dataset'
classes = sorted([d.name for d in Path(base_path).iterdir() if d.is_dir()])

filepaths = []
labels = []

for cls in classes:
    cls_path = Path(base_path) / cls
    for img_file in sorted(cls_path.glob('*.[jJ][pP]*')):  # jpg, jpeg, JPG, JPEG
        filepaths.append(str(img_file))
        labels.append(cls)

print(f"Total images: {len(filepaths)}")

# Create DataFrames
df = pd.DataFrame({'file_paths': filepaths, 'labels': labels})

# Split: 70% train, 15% valid, 15% test
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['labels'])
valid_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['labels'])

print(f"\nDataset split:")
print(f"  Train: {len(train_df)} images")
print(f"  Valid: {len(valid_df)} images")
print(f"  Test:  {len(test_df)} images")
print(f"\nClass distribution in training set:")
print(train_df['labels'].value_counts().sort_index())

## Step 2: Visualize Sample Images

In [ ]:
plt.figure(figsize=(15, 10))
for i in range(20):
    idx = np.random.randint(len(train_df))
    plt.subplot(4, 5, i+1)
    img = plt.imread(train_df.iloc[idx]['file_paths'])
    plt.imshow(img)
    plt.title(train_df.iloc[idx]['labels'], size=10, color='black')
    plt.xticks([])
    plt.yticks([])

plt.tight_layout()
plt.show()

## Step 3: Configure Data Augmentation & Generators

In [ ]:
# Configuration
target_size = (224, 224)
batch_size = 32

# Data augmentation for training
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    zoom_range=0.2,
    horizontal_flip=True
)

# No augmentation for validation/test (only preprocessing)
test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)

# Create data generators
train_gen = train_datagen.flow_from_dataframe(
    train_df,
    x_col='file_paths',
    y_col='labels',
    target_size=target_size,
    batch_size=batch_size,
    color_mode='rgb',
    class_mode='categorical'
)

valid_gen = test_datagen.flow_from_dataframe(
    valid_df,
    x_col='file_paths',
    y_col='labels',
    target_size=target_size,
    batch_size=batch_size,
    color_mode='rgb',
    class_mode='categorical'
)

test_gen = test_datagen.flow_from_dataframe(
    test_df,
    x_col='file_paths',
    y_col='labels',
    target_size=target_size,
    batch_size=batch_size,
    color_mode='rgb',
    class_mode='categorical'
)

# Save class indices for later use
class_indices = train_gen.class_indices
print(f"Class mapping: {class_indices}")

## Step 4: Build EfficientNetB0-based Model

In [ ]:
# Load pre-trained EfficientNetB0
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    input_shape=(224, 224, 3),
    weights='imagenet'
)

# Build model
num_classes = len(class_indices)
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

# Compile
lr = 0.001
model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=lr),
    metrics=['accuracy']
)

print(model.summary())

## Step 5: Configure Callbacks

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        'thai_food_model.h5',
        save_best_only=True,
        verbose=1,
        monitor='val_accuracy'
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        verbose=1,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        verbose=1,
        min_lr=1e-7
    )
]

print("Callbacks configured successfully")

## Step 6: Train the Model

In [ ]:
epochs = 30

history = model.fit(
    train_gen,
    validation_data=valid_gen,
    epochs=epochs,
    callbacks=callbacks,
    verbose=1,
    steps_per_epoch=len(train_gen),
    validation_steps=len(valid_gen)
)

print("Training complete!")

## Step 7: Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_title('Model Loss')
axes[0].set_ylabel('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Training Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[1].set_title('Model Accuracy')
axes[1].set_ylabel('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Print final metrics
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
print(f"\nFinal Training Accuracy: {final_train_acc:.4f}")
print(f"Final Validation Accuracy: {final_val_acc:.4f}")

## Step 8: Evaluate on Test Set

In [ ]:
# Load best model (Keras native format)
best_model = tf.keras.models.load_model('thai_food_model.keras')

# Evaluate on test set
test_loss, test_accuracy = best_model.evaluate(test_gen)
print(f"\nTest Set Evaluation:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# Get detailed predictions
predictions = best_model.predict(test_gen)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_gen.classes

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("\nClassification Report:")
class_names = list(class_indices.keys())
print(classification_report(true_classes, predicted_classes, target_names=class_names))

## Step 9: Export to TensorFlow.js Format

**Critical**: This is the step that avoids all Keras 3.x export issues! We use TensorFlow.js converter tool.

In [ ]:
# Step 9: Export to TensorFlow.js Format (Fixed - Graph Model approach)
import subprocess
import sys
import os
import shutil
from pathlib import Path
import json

print("=" * 80)
print("EXPORTING MODEL TO TENSORFLOW.JS FORMAT")
print("=" * 80)

# Step 1: Save as SavedModel first
saved_model_path = Path("thai_food_model_savedmodel")
print(f"\n1️⃣ Exporting to SavedModel format: {saved_model_path}")
if saved_model_path.exists():
    shutil.rmtree(saved_model_path)
best_model.export(str(saved_model_path))
print("✅ SavedModel created")

# Step 2: Convert SavedModel to GraphModel format (more compatible with TensorFlow.js)
output_dir = Path("tfjs_model")
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(exist_ok=True)

cmd = [
    "tensorflowjs_converter",
    "--input_format", "tf_saved_model",
    "--output_format", "tfjs_graph_model",
    str(saved_model_path),
    str(output_dir),
]
print(f"\n2️⃣ Converting SavedModel to GraphModel format...")
print("Command:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode != 0:
    print("⚠️ Conversion error output:")
    print(result.stderr)
    raise RuntimeError("Conversion failed")
else:
    print("✅ Conversion successful!")

print("\n3️⃣ Exported files:")
total_size = 0
for file in sorted(os.listdir(output_dir)):
    file_path = output_dir / file
    if file_path.is_file():
        file_size = file_path.stat().st_size / (1024 * 1024)
        total_size += file_size
        print(f"   - {file} ({file_size:.2f} MB)")
print(f"\n   Total size: {total_size:.2f} MB")
print("\n✅ Model exported successfully to tfjs_model/")

In [ ]:
# Save class mapping to JSON for use in browser
# CRITICAL: Create mapping sorted by index to ensure correct order
class_mapping = {v: k for k, v in class_indices.items()}  # Invert to index->name
print(f"DEBUG - class_indices: {class_indices}")
print(f"DEBUG - class_mapping (unsorted): {class_mapping}")

# Sort by index key to ensure correct order
sorted_class_names = [class_mapping[i] for i in sorted(class_mapping.keys())]
print(f"DEBUG - sorted_class_names: {sorted_class_names}")

with open('tfjs_model/class_names.json', 'w') as f:
    json.dump(sorted_class_names, f, indent=2)

print("\n✅ class_names.json created")
print(f"Classes (sorted by index): {sorted_class_names}")

## Step 10: Deployment Instructions

**Files to Deploy**:
- `tfjs_model/model.json` → Replace `public/model/model.json`
- `tfjs_model/group1-shard*.bin` → Replace shards in `public/model/`
- `tfjs_model/class_names.json` → Replace `public/model/class_names.json`

**After Deployment**:
1. Run `npm run dev`
2. Clear browser cache: **Ctrl+Shift+Delete** (or Cmd+Shift+Delete on Mac)
3. Visit the app and test food recognition
4. Open DevTools (F12) → Console to see logs

In [ ]:
print("""
═══════════════════════════════════════════════════════════════════════════════
                         TRAINING COMPLETE! ✅
═══════════════════════════════════════════════════════════════════════════════

📊 Model Performance:
   - Architecture: EfficientNetB0 + custom head
   - Input: 224×224 RGB images
   - Classes: {} Thai food dishes
   - Test Accuracy: {:.2f}%

📦 Export Location: tfjs_model/
   - model.json (model architecture)
   - group1-shard*.bin (model weights)
   - class_names.json (class mapping)

🚀 NEXT STEPS FOR DEPLOYMENT:
   1. Download all files from tfjs_model/ folder
   2. Copy to public/model/ in your React project
   3. Clear browser cache (Ctrl+Shift+Delete)
   4. Run: npm run dev
   5. Test the food recognition feature!

💡 TIP: This model uses TensorFlow 2.x compatible export, avoiding Keras 3.x issues!

═══════════════════════════════════════════════════════════════════════════════
""".format(num_classes, test_accuracy*100))